# 🔎 WikiKnowledge Explorer — Challenge 2

Explore Wikimedia articles, discover related content using TF-IDF and cosine similarity, and inspect structured information.


In [1]:
!pip -q install kaggle pandas numpy scikit-learn pyarrow


In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    'wikimedia-foundation/wikipedia-structured-contents',
    'enwiki/data/enwiki_namespace_0_00008.parquet'
)

print('Rows:', len(df))
print('Columns:', df.columns.tolist())


Rows: 25000
Columns: ['abstract', 'additional_entities', 'date_created', 'date_modified', 'description', 'event', 'identifier', 'image', 'in_language', 'infoboxes', 'is_part_of', 'license', 'main_entity', 'name', 'references', 'sections', 'tables', 'url', 'version']


In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

data = df.copy()
for column in ['name', 'abstract', 'description']:
    data[column] = data[column].fillna('').astype(str)
data = data[data['name'].str.strip() != ''].reset_index(drop=True)

print('=' * 70)
print('        WIKIKNOWLEDGE EXPLORER')
print('=' * 70)
print(f'\nWikimedia articles available: {len(data):,}')

search_query = input('\n🔎 Enter an article/topic to explore: ').strip()

if not search_query:
    print('\n❌ Please enter a valid search term.')
else:
    q = search_query.lower()
    exact = data[data['name'].str.lower() == q]
    starts = data[data['name'].str.lower().str.startswith(q)]
    contains = data[data['name'].str.lower().str.contains(q, regex=False, na=False)]
    matches = pd.concat([exact, starts, contains]).drop_duplicates('name').head(10).reset_index(drop=True)

    if matches.empty:
        print('\n❌ No matching articles found.')
    else:
        print(f'\n📌 Found {len(matches)} matching article(s):\n')
        for i, title in enumerate(matches['name'], 1):
            print(f'{i}. {title}')

        while True:
            choice = input(f'\nSelect article number (1-{len(matches)}): ').strip()
            if choice.isdigit() and 1 <= int(choice) <= len(matches):
                break
            print(f'❌ Please enter a number between 1 and {len(matches)}.')

        selected = matches.iloc[int(choice) - 1]
        print('\n' + '=' * 70)
        print('📖 SELECTED ARTICLE')
        print('=' * 70)
        print('\nTitle:', selected['name'])
        print('\nDescription:', selected['description'] or 'No description available.')
        print('\nAbstract:', selected['abstract'] or 'No abstract available.')

        data['combined_text'] = data['name'] + ' ' + data['description'] + ' ' + data['abstract']
        vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
        matrix = vectorizer.fit_transform(data['combined_text'])
        selected_position = data.index[data['name'] == selected['name']][0]
        scores = cosine_similarity(matrix[selected_position], matrix).flatten()
        related_indices = [i for i in scores.argsort()[::-1] if i != selected_position][:5]

        print('\n' + '=' * 70)
        print('🔗 RELATED ARTICLES')
        print('=' * 70)
        for number, index in enumerate(related_indices, 1):
            article = data.iloc[index]
            print(f'\n{number}. {article["name"]}')
            print(f'   Similarity: {scores[index] * 100:.1f}%')
            if article['description']:
                print(f'   Description: {article["description"][:300]}')
            if article['abstract']:
                print(f'   Information: {article["abstract"][:350]}')

        print('\n' + '=' * 70)
        print('🧩 STRUCTURED INFORMATION AVAILABLE')
        print('=' * 70)
        print('\n🔹 Main Entity:')
        print(str(selected.get('main_entity', 'No main entity available.'))[:1000])
        print('\n🔹 Additional Entities:')
        print(str(selected.get('additional_entities', 'No additional entities available.'))[:1500])
        print('\n📚 References:')
        print(str(selected.get('references', 'No references available.'))[:1000])

        explore = input('\nEnter related article number to explore (1-5), or press Enter to skip: ').strip()
        if explore.isdigit() and 1 <= int(explore) <= len(related_indices):
            related = data.iloc[related_indices[int(explore) - 1]]
            print('\n📖 RELATED ARTICLE DETAILS')
            print('Title:', related['name'])
            print('Description:', related['description'] or 'No description available.')
            print('Abstract:', related['abstract'][:1500] or 'No abstract available.')

        print('\n' + '=' * 70)
        print('🎯 WIKIKNOWLEDGE EXPLORER — COMPLETE')
        print('=' * 70)
        print(f'\nArticle explored: {selected["name"]}')
        print(f'Related articles identified: {len(related_indices)}')
        print('\n✅ Search\n✅ Article information retrieval\n✅ Related article discovery\n✅ Structured information\n✅ Article exploration')
